**NOTICE:**  
The U.S. Army Corps of Engineers, Risk Management Center (USACE-RMC) makes no guarantees about the results, or appropriateness of outputs, obtained from Numerics.

# 00. Getting Started with Numerics in Python
Welcome! This notebook will guide you through setting up and using the USACE-RMC Numerics library in Python via PythonNet.

## What You'll Learn

- What the Numerics library provides and where it fits in the workflow.
- How to obtain or build `Numerics.dll` and keep it up to date.
- How to load the correct .NET runtime and add the DLL reference from Python.
- How to run a quick sanity check and diagnose common setup issues.

## Virtual Python Environment
Before we get to Numerics, we need to ensure Python is up and running on your device and in these notebooks. In the top right hand corner of this notebook click on the **"Select Kernel"** button. A Python environment should be available for you to chose from, assuming you've downloaded Python. If you have Python on your computer, but cannot find a Python kernel to select, don't worry we have a solution. We will create and virtual Python environment just for this library to work with.

This virtual Python environment will give this library its own copy of Python, its own set of tools and libraries, and keeping it isolated from other projects. Before we start, please ensure you have the new version of Python installed locally on your device.

1. Open a new terminal and make sure it is pointing to your project folder (you should see the folder path in the prompt)
2. To create the virtual environment, run one of the commands  below:        
   ```bash 
   # Mac/Linx
   python3 -m venv .venv
   # Windows
   python -m venv .venv
   ```
   This will create the .venv folder in your project. Double check that you can see this folder in the file explorer.
3. Now we will activate the virtual environment:
   ```bash
   # Mac/Linx
   source .venv/bin/activate
   # Windows (Command Prompt)
   .venv\Scripts\activate.bat
   # Windows (PowerShell)
   .venv\Scripts\Activate.ps1
   ```
   You should now see `(.venv)` appear at the start of the terminal prompt. This means it's active!
4. Instal ipykernel. This is what allows Jupyter to see your virtual environment as a kernel. With the `.venv` active run:
   ```bash
   pip install ipykernel
   ```
   Then register it: 
   ```bash
   # Mac/Linx
   python3 -m ipykernel install --user --name=.venv --display-name "Python (.venv)"
   # Windows
   python -m ipykernel install --user --name=.venv --display-name "Python (.venv)"
   ```
5. Now you should be able to select this virtual environment as a kernel. Click on the **"Select Kernel"** in the top right. Chose Python environments and select Python(.venv), the one you just created! If it doesn't show up right away, try reloading your window.
6. Open the command palette (Crl+Shift+P or Cmd+Shift+P) and find **"Python:Select Interpreter"**. Chose the .venv option to tell VS code to always use this interpreter going forward.

Now we have ensured that we can use and run Python in there Jupyter notebooks. Time to get down to business in Numerics!

## What is Numerics?
Numerics is a comprehensive .NET library for numerical computing and statistical analysis, developed by the U.S. Army Corps of Engineers Risk Management Center [[1]](#1). It includes:

- 42+ probability distributions with advanced fitting methods
- Bayesian MCMC samplers (RWMH, DEMCz, HMC, and more)
- Machine learning algorithms (Random Forest, K-Means, GMM)
- Optimization routines (global and local)
- Statistical tests and bootstrap methods
- Time series analysis tools
- And more!

## Obtaining Numerics from the USACE-RMC GitHub

There are three recommended ways to get the Numerics library:

### Option 1: Download the Repository Directly

1. Visit the [USACE-RMC/Numerics GitHub repository](https://github.com/USACE-RMC/Numerics)
2. Click the green **"Code"** button and select **"Download ZIP"**
3. Extract the ZIP file to a location on your computer
4. Build the solution (requires Visual Studio or .NET SDK):
   ```bash
   cd Numerics
   dotnet build Numerics.sln --configuration Release
   ```
5. The compiled `Numerics.dll` will be in `Numerics/bin/Release/net8.0/` (or similar)

### Option 2: Clone with Git (Recommended for Auto-Updates)

This approach allows you to easily pull the latest changes without re-downloading everything.

1. Install Git (if not already installed): [git-scm.com](https://git-scm.com/)
2. Clone the repository:
   ```bash
   git clone https://github.com/USACE-RMC/Numerics.git
   cd Numerics
   ```
3. Build the DLL:
   ```bash
   dotnet build Numerics.sln --configuration Release
   ```
4. Update to latest version anytime:
   ```bash
   cd Numerics  # Navigate to the cloned directory
   git pull origin main  # Pull latest changes
   dotnet build Numerics.sln --configuration Release  # Rebuild
   ```

**Tip:** Create a shell script or batch file to automate the update and rebuild:

**Linux/Mac** (`update_numerics.sh`):
```bash
#!/bin/bash
cd /path/to/Numerics
git pull origin main
dotnet build Numerics.sln --configuration Release
echo "Numerics updated and rebuilt successfully!"
```

**Windows** (`update_numerics.bat`):
```batch
@echo off
cd C:\\path\\to\\Numerics
git pull origin main
dotnet build Numerics.sln --configuration Release
echo Numerics updated and rebuilt successfully!
```

### Option 3: NuGet Package

For the most convenient setup (no building required), use the NuGet package:
```bash
# Install the NuGet package to a local directory
nuget install RMC.Numerics -OutputDirectory ./packages
```

Then reference the DLL from `./packages/RMC.Numerics.{version}/lib/net8.0/Numerics.dll`. This demo was built off of verison 2 of the Numerics NuGet package.

## Prerequisites
Before running this notebook, ensure you have:
1. .NET Runtime (6.0+ or .NET Framework 4.8.1 on Windows)
2. Python 3.8+
3. Required packages (install via pip). See ```notebook-requirements.txt``` for the full list of external python packages you need to install to run all of the notebooks.

Now that you have Numerics up and running the core workflow is: 
1. Load the .NET runtime.
2. Add the Numerics DLL reference.
3. Import namespaces and use Numerics as if it were a native Python library.


## Step 1: Load the .NET Runtime
**CRITICAL:** You must load the .NET runtime BEFORE importing `clr`. This is the most common setup mistake.

Runtime selection:  
- Use `pythonnet.load("coreclr")` for .NET 6/7/8.  
- Use `pythonnet.load("netfx")` only for .NET Framework (Windows-only).  

If you're not sure which you have installed, check the Numerics build output folder (e.g., `net8.0` indicates CoreCLR).

In [1]:
import pythonnet

# Load the CoreCLR runtime (for .NET 6+)
# Use "netfx" instead if you're on Windows with .NET Framework
# If using .NET 4.8 skip this step!
pythonnet.load("coreclr")

print("✓ .NET runtime loaded successfully")

✓ .NET runtime loaded successfully


## Step 2: Import clr and Load Numerics DLL
Now we can import the CLR bridge and load the Numerics assembly.

Make sure your `dll_path` points to the DLL you actually built:
- `.../bin/Debug/net8.0/Numerics.dll` for Debug
- `.../bin/Release/net8.0/Numerics.dll` for Release

In [2]:
import clr 
from pathlib import Path 

# Path to your Numerics.dll
# MODIFY THIS PATH to make your installation
dll_path = Path(r"C:\GIT\Numerics\Numerics\bin\Debug\net8.0\Numerics.dll")

# Alternative: if Numerics.dll is in the same folder as this notebook:
# dll_path = Path("Numerics.dll")

if not dll_path.exists():
    raise FileNotFoundError(f"Cannot find Numerics.dll at {dll_path}")

clr.AddReference(str(dll_path))

print(f"✓ Loaded Numerics from: {dll_path}")

✓ Loaded Numerics from: C:\GIT\Numerics\Numerics\bin\Debug\net8.0\Numerics.dll


## Step 3: Import Numerics Namespaces
Now we can import classes from Numerics just like any other Python module. Be sure to check how the namespaces are organized within Numerics. Below we import the namespaces we need for a basic example using the Normal Distribution.

In [3]:
# Import the most commonly used namespaces
from Numerics.Distributions import Normal
from Numerics.Sampling.MCMC import LogLikelihood
from System import Array, Double, Func

print("✓ Numerics namespaces imported successfully")

✓ Numerics namespaces imported successfully


## Basic Example: Normal Distribution
Let's create a Normal Distribution and compute some basic statistics.

In [4]:
# Create a Normal Distribution with mean=100, sd=15
dist = Normal(100,15)

# Get statistical properties
print(f"Mean: {dist.Mean}")
print(f"Standard Deviation: {dist.StandardDeviation}")
print(f"Variance: {dist.Variance}")
print(f"Skewness: {dist.Skewness}")
print(f"Kurtosis: {dist.Kurtosis}")

print(f"\nSmall Sanity Check")
print(f"{Normal(0,1).CDF(0)} ~= 0.5 ✓")

Mean: 100.0
Standard Deviation: 15.0
Variance: 225.0
Skewness: 0.0
Kurtosis: 3.0

Small Sanity Check
0.5 ~= 0.5 ✓


## Working with .NET Arrays
Many Numerics methods require .NET arrays instead of Python lists. Here's how we can convert between them. There are also helper functions in `notebooks/helper_functions.py`: `convert_to_dotnet_array(pythonList)` and `convert_to_donet_2d_array(matrix)`.

In [11]:
# Python list to .NET Array[Double]
python_data = [10.5, 20.3, 15.7, 18.9, 22.1]
dotnet_array = Array[Double](python_data)

print(f"Python list: {python_data}")
# Accessing .NET array elements since Python doesn't know how to print a .NET array directly
print(f".NET array: {[dotnet_array[i] for i in range(dotnet_array.Length)]}")

# .NET array back to Python list
back_to_python = [dotnet_array[i] for i in range(dotnet_array.Length)]
# Or more concisely:
back_to_python = list(dotnet_array)

print(f"Converted back: {back_to_python}")

Python list: [10.5, 20.3, 15.7, 18.9, 22.1]
.NET array: [10.5, 20.3, 15.7, 18.9, 22.1]
Converted back: [10.5, 20.3, 15.7, 18.9, 22.1]


## Converting Python Functions to .NET Functions
Similar to how we must convert from Python lists to .NET arrays, we must also create .NET function to pass to Numerics' methods. The LogLikelihood function is heavily used in these notebooks, so we will take a look on how to create it as a Python function to wrap as .NET function. We will also look at how to wrap functions that have already been defined in Numerics that we can call into Python.

In [6]:
# Example data for log-likelihood
data = Array[Double]([12.0, 15.5, 14.2, 16.8, 13.9])

# Define Python log-likelihood function
def log_likelihood(params):
    dist = Normal(params[0], params[1])
    return dist.LogLikelihood(data)

# Wrap the Python function as a .NET Func
log_likelihood_func = LogLikelihood(log_likelihood)

print("✓ Log-likelihood function defined")

✓ Log-likelihood function defined


In Test_Numerics, we defined various simple functions for testing integration methods. Now we will pull one into this notebook, wrapping it in a .NET function so we can pass it back to a Numerics method.

In [7]:
# MODIFY this path to point to your Test_Numerics.dll
test_dll_path = Path(r"C:\GIT\Numerics\Test_Numerics\bin\Debug\net8.0\Test_Numerics.dll")

# Alternative: if Test_Numerics.dll is in the same folder as this notebook:
# dll_path = Path("Test_Numerics.dll")

if not test_dll_path.exists():
    raise FileNotFoundError(f"Cannot find Test_Numerics.dll at {test_dll_path}")

clr.AddReference(str(test_dll_path))

print(f"✓ Loaded Test_Numerics from: {test_dll_path}")

# This is importing from Test_Numerics.dll 
from Mathematics.Integration import Integrands

# Calling Integrands.FX3 creates a Python function, so we must wrap it
f = Func[Double, Double](Integrands.FX3) # Calling the function f(x) = x^3

print("✓ f(x) = x^3 defined")

✓ Loaded Test_Numerics from: C:\GIT\Numerics\Test_Numerics\bin\Debug\net8.0\Test_Numerics.dll
✓ f(x) = x^3 defined


## Common Problems and Solutions

### 1. Runtime Not Loaded Error
Make sure you load the runtime BEFORE importing clr.

```python
# X WRONG - will fail
import clr
import pythonnet
pythonnet.load("coreclr")

# ✓ CORRECT - load runtime FIRST
import pythonnet
pythonnet.load("coreclr")
import clr
```

### 2. DLL Not Found
Verify the path in `dll_path` is correct and points to a dll that exists. If no dll exists, be sure that you have built the Numerics solution.

### 3. Loading DLL More Than Once
When running multiple files at once, if they all load Numerics with ```clr.AddReference(str(dll_path))``` it may fail with an error that Numerics has already been loaded. If this happens, add the statement below to check the assemblies and only load Numerics if it is not already loaded. The code below doesn't output anything, but you'll know it worked if it runs without the failure error of loading multiple DLLS. This error shouldn't happened within these notebooks because they all have separate kernels. It would happen if you were trying to run multiple Python files all at once where each one loads the same DLL. In each of these files you would replace how you were originally loading the DLL with the chunk below.

```python
# Needed to access the assemblies
from System import AppDomain

assemblies = AppDomain.CurrentDomain.GetAssemblies()
assembly_names = [assembly.GetName().Name for assembly in assemblies]
if 'Numerics' not in assembly_names:
    clr.AddReference(str(dll_path))
```
Better safe than sorry!

### 4. Array Type Mismatch
Check that you're using .NET arrays where required.

```python
# X WRONG - Python list
data = [1, 2, 3]
dist.LogLikelihood(data)  # Error!

# ✓ CORRECT - .NET array
data = Array[Double]([1, 2, 3])
dist.LogLikelihood(data)  # Works!
```

### 5. Platform Issues
Use `pythonnet.load("netfx")` on Windows with .NET Framework.

### 6. Accessing .NET Properties
```python
# X WRONG - trying to call a property
mean_value = dist.Mean()  # Error!

# ✓ CORRECT - properties don't use parentheses
mean_value = dist.Mean
```

## Summary
In this notebook you:       
$\checkmark$ Located or built `Numerics.dll` and confirmed the expected output path     
$\checkmark$ Loaded the correct .NET runtime and added the Numerics assembly in Python      
$\checkmark$ Ran a basic example and reviewed common troubleshooting steps      

## Exercise
1. Build Numerics from source and verify the DLL path you will use
2. Restart the kernel and repeat the load sequence to confirm it works
3. Create a simple distribution and compute its `Mean` and `CDF`


## References

<a id="1">[1]</a> U.S. Army Corps of Engineers Risk Management Center, *Numerics Library Documentation*, 2026.
